In [ ]:
"""
Real-time inference and control loop for a trained Subway Surfers CNN agent.

This script loads a pretrained convolutional neural network and uses it
to perform live gameplay inference from screen captures, converting visual
frames into discrete in-game actions.

1. Model Loading
   - Loads `SubwayCNN` architecture and pretrained weights from checkpoint.
   - Switches model to evaluation mode (disables dropout / gradients).

2. Screen Capture & Preprocessing
   - Captures full-screen frames using MSS.
   - Converts raw RGB image into model-compatible tensor using shared
     preprocessing pipeline (`transform` from training script):
       • Crops the game region
       • Converts to grayscale
       • Resizes to 96x96
       • Converts to tensor
   - Adds batch dimension → [1, 1, 96, 96]

3. Action Prediction
   - Forward pass through CNN produces logits for 5 actions:
       0: roll, 1: left, 2: noop, 3: right, 4: jump
   - Softmax converts logits into probabilities.
   - Highest probability action is selected via argmax.

4. Confidence Filtering
   - If model confidence < 0.9, action is overridden to "noop"
     to reduce unstable or uncertain predictions.

5. Action Execution
   - Uses PyAutoGUI to simulate keyboard inputs:
       • down → roll
       • left/right → lane changes
       • up → jump
       • noop → no operation

6. Runtime Control
   - `P` toggles pause/resume of inference loop (with debounce delay)
   - `Q` safely exits the program
   - Small sleep interval (0.02s) controls execution frequency

This forms a closed-loop imitation learning system:
screen → CNN inference → action → game state update → repeat.
"""

In [1]:
import pyautogui
import mss
import numpy as np
from PIL import Image
from torchvision import transforms
from train_model import SubwayCNN, transform
import torch.nn.functional as F

12942


In [2]:
# play.py
import time
import torch

import keyboard


In [3]:

# ---------------------------
# ACTION MAP
# ---------------------------
ACTIONS = {
    0: "roll",
    1: "left",
    2: "noop",
    3: "right",
    4: "jump"
}


In [ ]:

# ---------------------------
# GAME REGION (your config)
# ---------------------------
screen_width = 1920
screen_height = 1080
width=400
height=500
height_add=150
game_region = {
    'width': width,
    'height': height,
    'left': (screen_width - width) // 2,
    'top': (screen_height + height_add - height) // 2
}

In [5]:


# ---------------------------
# LOAD MODEL
# ---------------------------
model = SubwayCNN(in_channels=1, num_actions=5)

checkpoint = torch.load("models/subway_cnn.pth", map_location="cpu")
model.load_state_dict(checkpoint["model_state"])

model.eval()

SubwayCNN(
  (backbone): Sequential(
    (0): Conv2d(1, 32, kernel_size=(8, 8), stride=(4, 4))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2))
    (3): ReLU()
    (4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    (5): ReLU()
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=4096, out_features=512, bias=True)
    (8): ReLU()
  )
  (head): Linear(in_features=512, out_features=5, bias=True)
)

In [6]:
def get_screen():
    with mss.mss() as sct:
        monitor = {
            "left": 0,
            "top": 0,
            "width": screen_width,
            "height": screen_height
        }
        screenshot = sct.grab(monitor)

    img = Image.frombytes("RGB", screenshot.size, screenshot.rgb)
    img = transform(img)  # applies crop, grayscale, resize

    # Ensure we have a tensor (in case transform doesn't include ToTensor)
    if not isinstance(img, torch.Tensor):
        img = transforms.ToTensor()(img)

    return img.unsqueeze(0)  # add batch dimension → [1, 1, 96, 96]

In [7]:

# ---------------------------
# ACTION EXECUTION
# ---------------------------
def perform_action(action):
    if action == 0:
        pyautogui.press("down")
    elif action == 1:
        pyautogui.press("left")
    elif action == 2:
        pass
    elif action == 3:
        pyautogui.press("right")
    elif action == 4:
        pyautogui.press("up")


In [ ]:

paused = False
noop_printed=False
while True:
    # ── hotkeys ──────────────────────────────────────────
    if keyboard.is_pressed("q"):
        print("Quitting...")
        break

    if keyboard.is_pressed("p"):
        paused = not paused
        print("Paused" if paused else "Resumed")
        time.sleep(0.3)  # debounce so one press doesn't toggle multiple times

    if paused:
        time.sleep(0.1)
        continue

    # ── inference ─────────────────────────────────────────
    frame = get_screen()

    with torch.no_grad():
        output = model(frame)
        probs = F.softmax(output, dim=1)[0]

    action = torch.argmax(probs).item()
    confidence = probs[action].item()

    # ── confidence gate ───────────────────────────────────
    if confidence < 0.9:
        action = 2  # noop
    action_name = ACTIONS[action]

    if action_name == "noop":
        if not noop_printed:
            print(f"{action_name} | confidence={confidence:.3f}")
            noop_printed = True
    else:
        noop_printed = False
        print(f"{action_name} | confidence={confidence:.3f}")
    perform_action(action)
    time.sleep(0.02)

C:\Users\gyawa\AppData\Local\Temp\ipykernel_11596\2088804432.py:2: DeprecationWarning: mss.mss is deprecated and will be removed in a future release; use mss.MSS instead
  with mss.mss() as sct:


noop | confidence=0.475
right | confidence=0.982
noop | confidence=0.485
left | confidence=0.993
left | confidence=0.999
noop | confidence=0.882
jump | confidence=0.923
noop | confidence=0.375
right | confidence=0.937
noop | confidence=0.451
jump | confidence=0.907
noop | confidence=0.434
left | confidence=0.908
noop | confidence=0.801
right | confidence=0.974
noop | confidence=0.382
right | confidence=0.972
noop | confidence=0.561
left | confidence=0.939
noop | confidence=0.779
roll | confidence=0.978
roll | confidence=0.994
noop | confidence=0.858
left | confidence=0.928
left | confidence=0.946
left | confidence=0.973
left | confidence=0.969
left | confidence=0.964
noop | confidence=0.469
right | confidence=0.904
noop | confidence=0.880
right | confidence=0.922
noop | confidence=0.889
right | confidence=0.906
noop | confidence=0.893
right | confidence=0.915
right | confidence=0.960
right | confidence=0.975
right | confidence=0.945
right | confidence=0.986
right | confidence=0.983
rig